<a href="https://colab.research.google.com/github/masterlyj/self-LLM_Agent_RL/blob/main/nb/Qwen3_(4B)_Instruct-QAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

要运行此笔记本，请在**免费**的 Tesla T4 Google Colab 实例上点击“*运行时*”，然后点击“*全部运行*”！
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 需要帮助请加入 Discord，并 ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上给我们点个星</i> ⭐
</div>

要在本地设备上安装 Unsloth，请参照[我们的指南](https://unsloth.ai/docs/get-started/install)。本笔记本遵循 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可。

您将学习如何进行[数据预处理](#Data)、如何[训练](#Train)、如何[运行模型](#Inference)以及如何保存模型。

### News

**隆重推出 Unsloth Studio**——一个全新的开源、无代码 Web UI，用于训练和运行大语言模型。 [博客](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio 训练界面"></a><br><sub><b>训练模型</b>——无需代码</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio 聊天界面"></a><br><sub><b>在 Mac、Windows 和 Linux 上</b>运行 GGUF 模型</sub></td>
</tr></table>

训练 MoE 模型——DeepSeek、GLM、Qwen 和 gpt-oss，速度提升 12 倍，显存占用减少 35%。[博客](https://unsloth.ai/docs/new/faster-moe)

超长上下文强化学习现已到来，上下文窗口扩大 7 倍！[博客](https://unsloth.ai/docs/new/grpo-long-context)

强化学习新动态：[FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

请访问我们的文档，了解所有[模型上传](https://unsloth.ai/docs/get-started/unsloth-model-catalog)和[Notebook](https://unsloth.ai/docs/get-started/unsloth-notebooks)。

### Installation

In [ ]:
%%capture

import os, re

# 判断是否在 Colab 环境中（通过环境变量中是否有 "COLAB_"）
if "COLAB_" not in "".join(os.environ.keys()):
    # 本地环境：直接 pip 安装 unsloth（它会自动处理依赖）
    !pip install unsloth
else:
    # Colab 环境：手动安装各个组件，并指定特定版本以避免冲突
    import torch
    # 提取 PyTorch 的主次版本号（如 2.10、2.9）
    v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)

    # 根据 PyTorch 版本选择对应的 xformers 版本
    xformers = 'xformers==' + {
        '2.10': '0.0.34',
        '2.9':  '0.0.33.post1',
        '2.8':  '0.0.32.post2'
    }.get(v, "0.0.34")  # 默认使用 0.0.34

    # 安装核心依赖（--no-deps 避免自动安装不兼容的依赖）
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    # 单独升级 torchao（量化相关）
    !pip install --no-deps --upgrade "torchao>=0.16.0"
    # 安装其他工具包
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer

# 以下部分在所有环境中都会执行（包括本地和 Colab），
# 但会进一步根据 PyTorch 版本强制重装特定版本的 torchao 和 fbgemm-gpu-genai

try:
    import torch
    _qat_torch_minor = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
except Exception:
    _qat_torch_minor = ""

# 版本映射字典
_qat_torchao_map = {
    "2.10": "0.16.0",
    "2.8":  "0.14.1",
    "2.9":  "0.15.0"
}
_qat_torchao = _qat_torchao_map.get(_qat_torch_minor, "0.16.0")  # 默认 0.16.0

_qat_fbgemm_map = {
    "2.10": "1.5.0",
    "2.8":  "1.3.0",
    "2.9":  "1.4.2"
}
_qat_fbgemm = _qat_fbgemm_map.get(_qat_torch_minor, "1.5.0")    # 默认 1.5.0

# 强制重新安装匹配的 torchao 和 fbgemm-gpu-genai（用于量化感知训练）
!pip install --upgrade --force-reinstall torchao=={_qat_torchao} fbgemm-gpu-genai=={_qat_fbgemm}

# 最后，安装指定版本的 transformers，并单独安装 trl（避免依赖冲突）
!pip install transformers==4.55.4 && pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# 预置的 4bit 量化模型列表（来自 Unsloth 官方 Hub）
fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # Qwen 14B 速度提升 2 倍
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit 动态量化，兼顾高精度和低显存占用
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [新] 支持 TTS 模型！
] # 更多模型请访问 https://huggingface.co/unsloth

# 加载模型和分词器（此处以 Qwen3-4B-Instruct 为例）
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",  # 模型名称或路径
    max_seq_length = 2048,   # 最大序列长度，可根据需要调整（支持长上下文）
    load_in_4bit = False,    # 是否使用 4bit 量化（可减少显存），此处设为 False
    load_in_8bit = False,    # [新!] 8bit 量化，精度略高，但显存占用约为 4bit 的 2 倍
    full_finetuning = False, # [新!] 是否进行全参数微调（而非 LoRA）
    # token = "YOUR_HF_TOKEN", # 如果需要访问门控模型，请填写 HuggingFace 的 access token
)

我们现在添加LoRA适配器，这样只需更新少量参数！

In [ ]:
# 使用 Unsloth 的 PEFT（参数高效微调）方法为模型添加 LoRA 适配器
# 返回一个可用于训练的 PEFT 模型对象
model = FastLanguageModel.get_peft_model(
    model,                       # 基础模型（已加载的 FastLanguageModel 实例）

    # LoRA 秩（rank），决定低秩矩阵的维度。常用值：8, 16, 32, 64, 128
    # 越大则参数量越多，表达能力越强，但显存占用和训练时间也会增加
    r = 16,

    # 指定要应用 LoRA 的目标模块（通常选择注意力层和 FFN 层的投影矩阵）
    # Qwen 等模型通常包含这些模块名
    target_modules = [
        "q_proj",   # 查询投影
        "k_proj",   # 键投影
        "v_proj",   # 值投影
        "o_proj",   # 输出投影
        "gate_proj",# 门控投影（用于 SwiGLU 等）
        "up_proj",  # FFN 上投影
        "down_proj",# FFN 下投影
    ],

    # LoRA 的缩放因子 alpha，通常 alpha = 2 * r 是一种常见设置
    # 用于调节 LoRA 输出的影响程度
    lora_alpha = 32,

    # LoRA 层的 dropout 概率。0 表示不使用 dropout（Unsloth 优化过，性能更好）
    lora_dropout = 0,

    # 偏置项的训练策略："none" 表示不训练偏置（最节省显存）
    bias = "none",

    # [新特性] 量化感知训练（QAT）方案，这里使用 "int4" 表示 4bit 量化
    # 结合 Unsloth 的优化，可节省约 30% 显存，支持更大的 batch size
    qat_scheme = "int4",

    # 梯度检查点（Gradient Checkpointing）策略
    # "unsloth" 是 Unsloth 的优化版本，适合超长上下文训练，以时间换显存
    # 也可以设为 True（使用标准实现）或 False（不使用）
    use_gradient_checkpointing = "unsloth",

    # 随机种子，确保结果可复现
    random_state = 3407,

    # 是否使用秩稳定 LoRA（Rank-Stabilized LoRA），一般设为 False
    use_rslora = False,

    # LoftQ 配置（一种量化 + LoRA 的初始化方法），这里不启用
    loftq_config = None,
)

### 1. `Unsloth: Applying QAT to mitigate quantization degradation`
- **QAT** = **Quantization-Aware Training**（量化感知训练）。  
  它是一种在训练过程中**模拟量化效果**的技术：让模型在训练时就适应低精度的表示，从而**在最终量化后仍能保持较高的准确率**，避免“先训练再量化”带来的性能大幅下降。
- 在 `get_peft_model` 中设置了 `qat_scheme = "int4"`，所以 Unsloth 会自动为该模型启用 QAT 初始化，并在后续微调中持续应用量化感知。

---

### 2. `patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers`
- 表示 Unsloth 识别了模型的 **36 层 Transformer 模块**（Qwen3-4B 通常有 36 层），并对每一层做了如下替换/包装：
  - **36 个 QKV 层**：将原始的 Q/K/V 投影矩阵替换为支持 QAT 的版本。
  - **36 个 O 层**：即输出投影（`o_proj`）也做了对应处理。
  - **36 个 MLP 层**：包括 `gate_proj`、`up_proj`、`down_proj` 等全连接前馈层，也都打上 QAT 补丁。
- 这些层的权重会以**量化友好的方式进行更新**，从而在最终使用 4bit 推理时获得更好的效果。

让我们检查一下是否启用了 QAT！

In [ ]:
# 遍历模型中的所有子模块（包括嵌套的层）
for module in model.modules():
    # 检查当前模块的类名中是否包含 "FakeQuantized"
    # FakeQuantized 是 QAT（量化感知训练）中用于模拟量化的层
    if "FakeQuantized" in module.__class__.__name__:
        # 如果找到了任何 FakeQuantized 层，说明 QAT 已成功应用
        print("QAT is applied!")
        break  # 找到后即可退出循环，无需继续检查

<a name="Data"></a>
### 数据准备

现在我们使用 `Qwen-3` 格式进行对话风格的微调。我们使用 [Maxime Labonne 的 FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) 数据集，该数据集采用 ShareGPT 风格。Qwen-3 的多轮对话格式如下所示：


```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```

我们使用 `get_chat_template` 函数来获取正确的对话模板。我们支持 `zephyr`、`chatml`、`mistral`、`llama`、`alpaca`、`vicuna`、`vicuna_old`、`phi3`、`llama3`、`phi4`、`qwen2.5`、`gemma3` 等更多模板。

In [ ]:
# 从 Unsloth 的聊天模板工具中导入获取聊天模板的函数
from unsloth.chat_templates import get_chat_template

# 使用 get_chat_template 为 tokenizer 设置 Qwen3 的对话格式模板
# 这样 tokenizer 就能自动将对话列表转换为模型所需的特殊 token 格式（如 <|im_start|> 等）
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",  # 指定使用 qwen3 的指令模板
)

In [ ]:
# 从 Hugging Face 的 datasets 库导入加载数据集的函数
from datasets import load_dataset

# 加载 mlabonne/FineTome-100k 数据集，并指定使用训练集（split = "train"）
dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

`mlabonne/FineTome-100k` 是一个高质量的**指令微调数据集**，主要用于**提升大语言模型（LLM）的对话和推理能力**。

它的核心信息如下：

| 项目 | 说明 |
| :--- | :--- |
| **数据规模** | 包含 **10万** 个高质量的“指令-回复”对。 |
| **数据格式** | 采用 **ShareGPT** 格式，适用于多轮对话场景的微调。 |
| **主要用途** | 用于模型的**指令微调（Instruction Tuning）**，让模型更好地理解和遵循人类的指令。 |
| **设计目标** | 教会模型进行**细致的推理**、**基于事实的回应**以及**自然的对话**。 |
| **许可证** | 仅限**研究和教育用途**。 |


1.  **源头**：它源自一个更大的数据集 `arcee-ai/The-Tome`。
2.  **筛选**：创建者从中剔除了部分由特定模型生成的数据，并使用了 Hugging Face 的 `fineweb-edu-classifier` 进行了**重新筛选**，以确保留下的是高质量、教育意义强的样本。

我们现在使用 `standardize_data_formats` 来尝试将数据集转换为适合微调的正确格式！

In [ ]:
# 将数据集统一标准化为 Unsloth 支持的通用对话格式（如 ShareGPT 格式）
# 这样可以兼容不同来源的数据集（Alpaca、ShareGPT、自定义格式等）
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

来看看第100行是什么样子吧！

In [ ]:
dataset[100]

我们现在需要将 `Qwen-3` 的聊天模板应用到对话中，并将其保存到 `text` 字段。

In [ ]:
# 定义一个函数，用于将数据集中的对话格式化为模型训练所需的文本
def formatting_prompts_func(examples):
    # 从样本中提取对话列表（conversations 字段）
    convos = examples["conversations"]

    # 使用 tokenizer 的聊天模板将每个对话转换为文本字符串
    # tokenize=False 表示不进行 tokenize，只返回字符串
    # add_generation_prompt=False 表示不添加生成提示（如 <|im_start|>assistant 等）
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]

    # 返回一个字典，包含新字段 "text"，其值为转换后的文本列表
    return {"text": texts}

# 使用 map 方法将格式化函数应用到整个数据集
# batched=True 表示批量处理，以提高效率
dataset = dataset.map(formatting_prompts_func, batched=True)

来看看聊天模板的效果如何！

In [ ]:
dataset[100]['text']

<a name="Train"></a>
### 训练模型
现在我们来训练模型。为了加快速度，这里只执行 60 步；若要进行完整训练，可将 `num_train_epochs` 设为 1，并将 `max_steps` 设为 `None`。

In [ ]:
from trl import SFTTrainer, SFTConfig

# 创建 SFTTrainer（监督微调训练器）实例
trainer = SFTTrainer(
    model = model,                 # 已加载并添加 LoRA 的模型（PEFT 模型）
    tokenizer = tokenizer,             # 已设置聊天模板的分词器
    train_dataset = dataset,            # 训练数据集（已格式化为 "text" 字段）
    eval_dataset = None,              # 可选的验证数据集（此处不设置）

    # 训练配置参数（SFTConfig 对象）
    args = SFTConfig(
        # 数据集中的文本字段名，用于提取输入文本
        dataset_text_field = "text",

        # 每个设备的训练批次大小（因显存限制设为 1）
        per_device_train_batch_size = 1,

        # 梯度累积步数：每 4 步更新一次权重，等效批次大小 = 1 * 4 = 4
        gradient_accumulation_steps = 4,

        # 预热步数（学习率从 0 逐渐上升到设定值所需的步数）
        warmup_steps = 5,

        # num_train_epochs = 1,        # 将此设置为完成一次完整的训练运行。
        # 训练总步数（设为 30 步，用于快速演示；完整训练可注释掉并用 num_train_epochs）
        max_steps = 30,

        # 学习率（常用范围：1e-5 ~ 5e-4；长时间训练建议降至 2e-5）
        learning_rate = 2e-4,

        # 日志记录间隔（每 1 步输出一次损失值）
        logging_steps = 1,

        # 优化器（8-bit AdamW，节省显存）
        optim = "adamw_8bit",

        # 权重衰减系数（L2 正则化，防止过拟合）
        weight_decay = 0.001,

        # 学习率调度器类型（线性衰减）
        lr_scheduler_type = "linear",

        # 随机种子（确保可复现性）
        seed = 3407,

        # 日志报告目标（"none" 表示不向外部服务如 WandB 报告）
        report_to = "none",
    ),
)

我们还使用Unsloth的`train_on_completions`方法，仅对助手输出进行训练，并忽略用户输入部分的损失。这有助于提升微调的准确性！现在Unsloth会自动从分词器的聊天模板中识别指令与回复部分，因此无需再手动传入`instruction_part`和`response_part`。若你使用的是自定义聊天模板，仍可显式指定这两个参数。

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)

我们来验证指令部分是否已成功屏蔽！再打印第100行看看。

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

现在我们来打印经过掩码处理后的示例——你应该只会看到答案显示出来。

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
# @title Show current memory stats
# 获取当前 GPU 的设备属性（如名称、总显存等）
gpu_stats = torch.cuda.get_device_properties(0)

# 计算当前已保留（reserved）的显存峰值（单位：GB）
# max_memory_reserved() 返回当前进程已分配的显存字节数（包含缓存）
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

# 获取 GPU 的总显存大小（单位：GB）
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

# 打印 GPU 型号和总显存
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")

# 打印当前已保留的显存大小（这通常是在模型加载、数据预处理后但训练开始前的状态）
print(f"{start_gpu_memory} GB of memory reserved.")

开始训练模型吧！若要恢复训练任务，请设置 `trainer.train(resume_from_checkpoint=True)`。

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
# 获取训练结束后（或峰值）已保留的显存（单位：GB）
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

# 计算训练本身额外占用的显存（扣除加载模型后初始占用）
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)

# 计算已用显存占总显存的百分比
used_percentage = round(used_memory / max_memory * 100, 3)

# 计算训练额外占用显存的百分比
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

# 打印训练总耗时（秒）
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")

# 打印训练总耗时（分钟）
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)

# 打印峰值显存占用（总）
print(f"Peak reserved memory = {used_memory} GB.")

# 打印训练过程额外占用的峰值显存
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")

# 打印峰值显存占总显存的百分比
print(f"Peak reserved memory % of max memory = {used_percentage} %.")

# 打印训练额外显存占总显存的百分比
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

训练完成后，我们将 FakeQuantizedLinear 层转换回标准的 nn.Linear 层。这一步可以消除伪量化带来的额外开销，并为模型的最终转换或 LoRA 适配器合并做好准备。

In [ ]:
# 从 torchao 库导入量化函数和 QAT 配置
from torchao.quantization import quantize_
from torchao.quantization.qat import QATConfig

# 对模型应用量化转换（从 QAT 训练模式切换到推理量化模式）
# step="convert" 表示将训练时的伪量化（FakeQuant）替换为真正的量化权重，
# 以便模型能在推理时以低精度（如 int4）高效运行
quantize_(model, QATConfig(step="convert"))

<a name="Inference"></a>
### 推理
让我们通过 Unsloth 原生推理来运行模型！根据 `Qwen-3` 团队的推荐，指令类推理的推荐参数为 `temperature = 0.7, top_p = 0.8, top_k = 20`。

对于基于推理对话的场景，推荐使用 `temperature = 0.6, top_p = 0.95, top_k = 20`。

In [ ]:
messages = [
    {"role" : "user", "content" : "继续数列：1, 1, 2, 3, 5, 8,"}
]
# 使用 tokenizer 的聊天模板将消息列表转换为模型可接受的文本格式
# tokenize=False 表示返回字符串而非 token ID
# add_generation_prompt=True 表示在末尾添加 <|im_start|>assistant 提示符，以引导模型生成回复
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # 生成时必须设为 True
)

# 导入文本流式输出器，用于实时显示生成内容
from transformers import TextStreamer

# 调用模型的 generate 方法进行文本生成
# 使用下划线 _ 忽略返回的完整输出，因为 streamer 会实时打印
_ = model.generate(
    # 将文本 tokenize 并转换为 PyTorch 张量，移动到 GPU
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens=1000,            # 最大生成 token 数，可调大以获取更长输出
    temperature=0.7,              # 控制随机性，越低越确定
    top_p=0.8,                 # 核采样概率阈值
    top_k=20,                 # 限制最高概率的 top-k 个 token
    streamer=TextStreamer(tokenizer, skip_prompt=True),  # 流式输出，跳过输入提示
)

<a name="Save"></a>
### 保存与加载微调模型
若要将最终模型保存为 LoRA 适配器，可使用 Hugging Face 的 `push_to_hub` 进行在线保存，或使用 `save_pretrained` 进行本地保存。

**[注意]** 此方法仅保存 LoRA 适配器，而非完整模型。如需保存为 16bit 格式或 GGUF 格式，请向下滚动查看！

In [ ]:
# 将训练好的 LoRA 适配器权重保存到本地目录 "qwen_lora"
model.save_pretrained("qwen_lora")  # 保存模型（包含 LoRA 权重和配置）

# 将对应的分词器保存到同一目录，确保推理时使用相同的词汇表和模板
tokenizer.save_pretrained("qwen_lora")

# （可选）将模型推送到 Hugging Face Hub，供他人或自己在线使用
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN")  # 需要 Hugging Face 访问令牌

# （可选）将分词器也推送到 Hub，与模型配套
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN")

现在，如果你想加载我们刚刚保存的LoRA适配器用于推理，请将`False`改为`True`：

In [ ]:
# 这是一个示例代码块（当前被禁用），用于从本地加载之前保存的 LoRA 模型
if True:
    from unsloth import FastLanguageModel

    # 从本地目录 "qwen_lora" 加载微调后的模型和分词器
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="qwen_lora",        # 保存模型的本地路径
        max_seq_length=2048,           # 最大序列长度，应与训练时一致
        load_in_4bit=True,             # 以 4bit 量化加载，节省显存
    )

我们现在可以使用 TorchAO 保存并对最终模型进行量化，采用与 QAT 训练期间相同的配置。

In [ ]:
# 使用 torchao 的保存方法将量化后的模型保存到本地目录 "model"
# 该方法会同时保存模型权重（已转为真正的量化格式，如 int4）和分词器配置
model.save_pretrained_torchao(
    "model",   # 保存目录名
    tokenizer, # 分词器也会一并保存到同一目录
    torchao_config=QATConfig(step="convert")
)

### TorchAO 导出与转换

我们还支持使用自定义配置将模型导出为 TorchAO 量化检查点，以便在 vLLM 或其他推理引擎中进行推理。

如需深入了解 TorchAO 配置，可参考 Hugging Face Transformers 官方文档：https://huggingface.co/docs/transformers/main/quantization/torchao

In [ ]:
# 以下代码块默认被禁用（if False），仅供参考，需要时请改为 True 并执行

# --------------------------------------------------------------
# 保存为 TorchAO int4 格式（本地）
# --------------------------------------------------------------
if False:
    from torchao.quantization import Int4WeightOnlyConfig  # 导入 int4 仅权重量化配置

    # 将当前模型保存为 int4 量化格式（仅权重量化），并同时保存分词器
    model.save_pretrained_torchao(
        "model",                                     # 本地保存目录
        tokenizer,
        torchao_config=Int4WeightOnlyConfig()        # 使用 int4 配置（仅权重量化）
    )

# --------------------------------------------------------------
# 推送 int4 模型到 Hugging Face Hub（在线）
# --------------------------------------------------------------
if False:
    from torchao.quantization import Int4WeightOnlyConfig

    model.save_pretrained_torchao(
        "HF_USERNAME/model",                         # 替换为你的 Hugging Face 用户名和仓库名
        tokenizer,
        torchao_config=Int4WeightOnlyConfig(),       # int4 配置
        push_to_hub=True,                            # 启用推送到 Hub
        token="YOUR_HF_TOKEN"                        # 请替换为你自己的 Hugging Face 访问令牌（在 https://huggingface.co/settings/tokens 获取）
    )

# --------------------------------------------------------------
# 保存为 TorchAO int8 格式（本地）
# --------------------------------------------------------------
if False:
    from torchao.quantization import Int8DynamicActivationInt8WeightConfig  # int8 动态激活 + int8 权重配置

    model.save_pretrained_torchao(
        "model",                                     # 本地保存目录
        tokenizer,
        torchao_config=Int8DynamicActivationInt8WeightConfig()  # 使用 int8 动态激活 + int8 权重配置
    )

# --------------------------------------------------------------
# 推送 int8 模型到 Hugging Face Hub（在线）
# --------------------------------------------------------------
if False:
    from torchao.quantization import Int8DynamicActivationInt8WeightConfig

    model.save_pretrained_torchao(
        "HF_USERNAME/model",                         # 替换为你的用户名和仓库名
        tokenizer,
        torchao_config=Int8DynamicActivationInt8WeightConfig(),  # int8 配置
        push_to_hub=True,                            # 启用推送到 Hub
        token="YOUR_HF_TOKEN"                        # 请替换为自己的访问令牌
    )

好了，我们完成了！如果你对 Unsloth 有任何疑问，可以加入我们的 [Discord](https://discord.gg/unsloth) 频道！如果你发现任何 Bug，或者想了解最新的 LLM 动态，或者需要帮助、参与项目等，欢迎随时加入我们的 Discord！

其他一些资源：
1. 想在本地使用 Unsloth？请阅读我们的[安装指南](https://unsloth.ai/docs/get-started/install)，了解如何在 Windows、Docker、AMD、Intel GPU 上安装 Unsloth。
2. 通过我们的[强化学习指南和笔记本](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide)学习如何进行强化学习。
3. 阅读我们的[文本转语音（TTS）](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning)和[视觉](https://unsloth.ai/docs/basics/vision-fine-tuning)模型支持指南和笔记本。
4. 浏览我们的 [LLM 教程目录](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms)，查找每个模型的专属指南。
5. 推理需要帮助？请阅读我们的[推理与部署页面](https://unsloth.ai/docs/basics/inference-and-deployment)，了解如何使用 vLLM、llama.cpp、Ollama 等。

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  需要帮助请加入 Discord，并 ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们点星</i> ⭐️

  本笔记本及所有 Unsloth 笔记本均遵循 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可
</div>